# 使用 LLM 进行人工智能支持的 AC 竞争对手分析

## 问题陈述
在当今竞争激烈的市场中，由于公司网站上提供的信息分散且非结构化，客户和企业常常难以比较不同品牌的产品。

该项目旨在通过构建一个**人工智能驱动的系统**来解决该问题，该系统使用网站数据和大型语言模型（LLM）自动分析和比较三星和松下等领先品牌的空调（AC）。

## 目标

开发一个系统：
- 从公司网站提取产品相关数据  
- 确定主要功能、定价和规格  
- 使用法学硕士生成结构化比较

In [ ]:
#进口
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

class ACComparator:
    def __init__(self, model="llama3.2:1b"):
        self.model = model
        self.headers = {
            "User-Agent": "Mozilla/5.0"
        }

    def smart_fetch(self, url):
        try:
            res = requests.get(url, headers=self.headers, timeout=10)
            return res.text
        except:
            return ""
    
    def get_text(self, url):
        html = self.smart_fetch(url)
        soup = BeautifulSoup(html, "html.parser")

        for tag in soup(["script", "style"]):
            tag.decompose()

        return soup.get_text(separator="\n", strip=True)
    
    def call_llm(self, prompt, system_prompt="You are a helpful assistant"):
        from openai import OpenAI

        client = OpenAI(
            base_url="http://localhost:11434/v1",
            api_key="ollama"
        )

        response = client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ]
        )

        return response.choices[0].message.content
    
    def generate_AC_Comparison(self, name1, data1, name2, data2):
        prompt = f"""
        Compare {name1} and {name2} Air Conditioners.

        DATA {name1}:
        {data1[:3000]}

        DATA {name2}:
        {data2[:3000]}

        Generate:
        1. Comparison table (Price, Features, Energy Efficiency)
        2. Strengths of each
        3. Which is better for Indian users
        4. Final recommendation
        """

        return self.call_llm(prompt)

In [ ]:
analyst = ACComparator()

# 交流页
samsung_url = "https://www.samsung.com/in/air-conditioners/"
panasonic_url = "https://store.in.panasonic.com/air-conditioners.html"

# 获取内容
samsung_data = analyst.get_text(samsung_url)
panasonic_data = analyst.get_text(panasonic_url)

# 生成比较
result = analyst.generate_AC_Comparison(
    "Samsung",
    samsung_data,
    "Panasonic",
    panasonic_data
)

print(result)

## 结论

该项目展示了法学硕士如何与网络抓取相结合，将非结构化网络数据转化为有意义的业务洞察，从而实现更智能、更快速的决策。